Desinging a straightforward task-array workflow on Aire.

I created the CSV "wales_filenames" that point to specific files in the folder "parks_gardens_id". In this notebook, I'm going to create a "dry run" version of the workflow, and then will put it in a Python script, and add it to Aire

In [19]:
import pandas as pd
import geopandas as gpd
import os
from pathlib import Path
import sys

sys.path.insert(0, '../src')

import park_vga

In [13]:
# open the filenames csv
files_list = pd.read_csv("wales_filenames.csv")

data_folder = "parks_gardens_id/"

#example output folder, will be deleted after the workflow is run
output_folder = "aire_runs/wales_aire_run_output"
os.makedirs(output_folder, exist_ok=True)

wales_dtm = "/Volumes/Extreme SSD/wales_lidar/wales_dtm_32bit_cog.tif"
wales_dsm = "/Volumes/Extreme SSD/wales_lidar/wales_dsm_32bit_cog.tif"

park_ids_file = "../example_datasets/all_parks_ids.csv"
check_regions_file = "../example_datasets/LUT_regions_authorities_filenames.geojson"

In [14]:
files_list

,filename
0,Swansea_pp_or_g_cmb.geojson
1,Rhondda Cynon Taf_pp_or_g_cmb.geojson
2,Denbighshire_pp_or_g_cmb.geojson
3,Torfaen_pp_or_g_cmb.geojson
4,Vale of Glamorgan_pp_or_g_cmb.geojson
5,Powys_pp_or_g_cmb.geojson
6,Carmarthenshire_pp_or_g_cmb.geojson
7,Bridgend_pp_or_g_cmb.geojson
8,Blaenau Gwent_pp_or_g_cmb.geojson
9,Cardiff_pp_or_g_cmb.geojson


In [20]:
# this portion of the loop, running through the filenames, won't happen
# in python - this is part of the slurm workflow (parallel task array)

for filename in files_list.filename:

    file_path = data_folder + filename # this is essentially the bit to be handled by slurm
    gdf = gpd.read_file(file_path)
    output_sub_dir = os.path.join(output_folder, filename.split(".")[0])
    os.makedirs(output_sub_dir, exist_ok=True)
    # run the aire workflow for this park, with the file_path as input, and output_sub_dir as output
    # the workflow will also need the wales_dtm, wales_dsm, park_ids_file, and check_regions_file as inputs - these can be passed as arguments to the workflow script
    check_regions_gdf = gpd.read_file(check_regions_file)
    check_regions_gdf["filename"] = check_regions_gdf["filename"].apply(lambda x: Path(x).name)
    country = check_regions_gdf.loc[check_regions_gdf["filename"] == filename, "country"].values[0]
    authority = check_regions_gdf.loc[check_regions_gdf["filename"] == filename, "auth_name_e"].values[0]
    print(country, authority)


    # filter the park ids dataframe to just the authority
    park_ids_file_df = pd.read_csv(park_ids_file)
    # subset to where auth_name_e matches authority
    park_ids_file_df = park_ids_file_df.loc[park_ids_file_df["auth_name_e"] == authority]
    print(len(park_ids_file_df))
    for n in range(len(gdf)):
        print(f"{n}/{len(gdf)}")
        park_id = gdf.loc[n, "id"]
        print(park_id)
        # use subsetted park ids dataframe to find the row where park_id matches id in park_ids_file_df
        # and pull out new park id (safe for filenames)
        park_id_safe = park_ids_file_df.loc[park_ids_file_df["old_park_id"] == park_id, "new_park_id"].values[0]
        print(park_id_safe)
        # check if output files already exist for this park, and if so, skip to the next park
        output_file = output_sub_dir + f"/{park_id_safe}_visibility.geojson"
        if Path(output_file).exists():
            print(f"Output file {output_file} already exists, skipping park {park_id_safe}")
            continue
        if country == "Wales":
            dtm_path = wales_dtm
            dsm_path = wales_dsm
            print(file_path)
            print(output_sub_dir)
            # results = park_vga.workflow.workflow_wales(file_path, n,
            #                                     dtm_path, dsm_path,
            #                                     output_sub_dir,
            #                                     spacing=12,
            #                                     return_results=False, save_results=True,
            #                                     park_id_for_file_name=park_id_safe,
            #                                     max_distance=80)



    
    


Wales Swansea
1255
0/1255
SWANS_841e2e988f55;SWANS_9add957b9b60
3816d9f14391
parks_gardens_id/Swansea_pp_or_g_cmb.geojson
aire_runs/wales_aire_run_output/Swansea_pp_or_g_cmb
1/1255
SWANS_64180ad89ee2;SWANS_7875d4152bbb
eeb8a45abba6
parks_gardens_id/Swansea_pp_or_g_cmb.geojson
aire_runs/wales_aire_run_output/Swansea_pp_or_g_cmb
2/1255
SWANS_0077e0499336;SWANS_003142c8af8d
2afa4425d6cc
parks_gardens_id/Swansea_pp_or_g_cmb.geojson
aire_runs/wales_aire_run_output/Swansea_pp_or_g_cmb
3/1255
SWANS_8c5a5b68c7e1;SWANS_fb7bfbb4dae0
ee93e1ee473b
parks_gardens_id/Swansea_pp_or_g_cmb.geojson
aire_runs/wales_aire_run_output/Swansea_pp_or_g_cmb
4/1255
SWANS_8119996c470b
a7f47d7bf887
parks_gardens_id/Swansea_pp_or_g_cmb.geojson
aire_runs/wales_aire_run_output/Swansea_pp_or_g_cmb
5/1255
SWANS_a6bcbbfa6044
357bcc16acd2
parks_gardens_id/Swansea_pp_or_g_cmb.geojson
aire_runs/wales_aire_run_output/Swansea_pp_or_g_cmb
6/1255
SWANS_09b85b3feb42;SWANS_76a8b8757e67
a42c15a731f7
parks_gardens_id/Swansea_pp_or_